In [0]:
%sql
USE flights_bronze;

### Załadowanie danych źródłowych

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.types import *

airports_schema = StructType([
    StructField('iata_code', StringType(), True),
    StructField('airport', StringType(), True),
    StructField('city', StringType(), True),
    StructField('state', StringType(), True),
    StructField('country', StringType(), True),
    StructField('latitude', DoubleType(), True),
    StructField('longitude', DoubleType(), True)
])

airports_bronze = "flights_bronze.airports"

df_airports = spark.read.format("csv") \
    .schema(airports_schema) \
    .option("header", "true") \
    .load("/Volumes/workspace/hurtownie_danych/cw7-projekt/airports.csv")

df_airports.write.mode("overwrite").saveAsTable(airports_bronze)

In [0]:
%python
display(df_airports.limit(5))

iata_code,airport,city,state,country,latitude,longitude
ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.4404
ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.6819
ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447


### Proces ładowania przyrostowego

**Logika *MERGE* dla SCD1 - nadpisanie danych**  
Tabela *airports* to tabela wymiaru pisująca atrybuty portów lotniczych. Jeśli cokolwiek, np. nazwa lotniska lub miasto, się zmieni, wtedy nadpisaujemy stare dane. Nie musimy martwić się jednak o utratę danych historycznych przekazanych do warstwy silver, ponieważ tam następuje osobny prosces ładowania przyrostowego, odpowiedni dla SCD2.

In [0]:
%python

airports_bronze = "flights_bronze.airports"
new_file_path = "/Volumes/workspace/hurtownie_danych/cw7-projekt/airports_new.csv"

# sprawdzamy czy nowy plik istnieje
file_exists = False
try:
  dbutils.fs.ls(new_file_path) # rzuci wyjątek, jeśli nie istnieje
  file_exists = True
except Exception as e:
  if "No such file or directory" in str(e):
    pass
  else:
    raise e

# jeśli istnieje - wczytujemy
if file_exists:
  df_airlines_new = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(new_file_path)

  # jeśli tabela nie istnieje, wykonujemy pierwszy pełen zapis, a jeśli istnieje, robimy MERGE INTO
  if not spark.catalog.tableExists(airports_bronze):
    df_airlines_new.write.format("delta").saveAsTable(airports_bronze)
  else:
    df_airlines_new.createOrReplaceTempView("src_airports")

    spark.sql(f"""
      MERGE INTO {airports_bronze} AS ap
      USING src_airports AS s
      ON ap.iata_code = s.iata_code
      WHEN MATCHED THEN
        UPDATE SET 
        ap.airport = s.airport,
        ap.city = s.city,
        ap.state = s.state,
        ap.country = s.country,
        ap.latitude = s.latitude,
        ap.longitude = s.longitude
      WHEN NOT MATCHED THEN
        INSERT *
    """)
else:
  print(f"File {new_file_path} does not exist or have been already processed.")


File /Volumes/workspace/hurtownie_danych/cw7-projekt/airports_new.csv does not exist or have been already processed.
